# MLflow basics: track, compare, register

Airflow coordinates tasks, schedules work, and helps recover from failures. MLflow records training results and manages model versions. We need both repeatable execution and a traceable model lifecycle.

**Problem:** we have several model approaches. Manual training makes consistent comparisons and tracing the chosen model difficult.

**Today's solution:** load Iris, train two simple classifiers, track both, compare accuracy, register the winner, and assign `candidate`. Airflow will orchestrate these same steps later.

## 1. Verify the environment
Select the `mlops-workshop-env` kernel in VS Code. Run this notebook from inside the project; its setup finds the project root automatically. Imports and versions below confirm that the kernel has the required packages.

In [ ]:
# Standard Python tools: inspect the environment and work with file paths.
import sys
from pathlib import Path

# MLflow saves training records and model files.
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient
from mlflow.models import infer_signature

# pandas displays our comparison table; scikit-learn provides data and models.
import pandas as pd
import sklearn
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Check which Python environment is running this notebook.
print("Interpreter:", sys.executable)
print("Python:", sys.version.split()[0])
print("MLflow:", mlflow.__version__)
print("scikit-learn:", sklearn.__version__)
print("pandas:", pd.__version__)

## 2. MLflow vocabulary
MLflow Tracking records how a model was trained and how it performed. Its Model Registry keeps named model versions and aliases.

| Term | Meaning in this notebook |
| --- | --- |
| Experiment | A group of related runs: `iris-model-comparison` |
| Run | One training attempt, such as Logistic Regression |
| Parameter | A setting, such as `max_iter` or the split seed |
| Metric | A measured result: held-out accuracy |
| Artifact | A saved file, such as the trained model and its input schema |

We log explicitly so each operation is visible. SQLite supports both tracking and the registry locally; no server is needed for training.

### Find the project folder
We keep the database and artifacts in the project root, even if the notebook starts inside `notebooks/`.

In [ ]:
# Find the project root from this notebook's folder or any project subfolder.
project_root = Path.cwd().resolve()
while not (project_root / "requirements.txt").is_file():
    if project_root == project_root.parent:
        raise RuntimeError("Open the notebook from inside the workshop project.")
    project_root = project_root.parent

### Configure tracking
The tracking URI tells MLflow where to save its records. We use one local SQLite database for tracking and the model registry.

In [ ]:
# Build an absolute database path that works on Windows, Linux, and macOS.
tracking_uri = "sqlite:///" + (project_root / "mlflow.db").as_posix()

# Use this database for training records and registered model versions.
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_registry_uri(tracking_uri)

# The client lets us read and manage MLflow records.
client = MlflowClient()
print("Tracking database:", tracking_uri)

### Select the experiment
An experiment groups related training runs. Create it once, then reuse it when the notebook runs again.

In [ ]:
experiment_name = "iris-model-comparison"

# Create the experiment only if it does not exist yet.
if client.get_experiment_by_name(experiment_name) is None:
    client.create_experiment(
        experiment_name,
        artifact_location=(project_root / "mlartifacts").as_uri(),
    )

# Both model runs will belong to this experiment.
experiment = mlflow.set_experiment(experiment_name)
print("Experiment:", experiment.name)

## 3. Load a tiny dataset
Iris is bundled with scikit-learn: 150 flowers, four numeric features, and three classes. No download, ETL, or feature engineering is required.

In [ ]:
# Iris is included with scikit-learn, so no download is needed.
iris = load_iris(as_frame=True)

# X contains the four measurements; y contains the flower classes.
X = iris.data.astype("float64")
y = iris.target

print("Rows and features:", X.shape)
print("Classes:", iris.target_names.tolist())
X.head()

## 4. Make one shared train/test split
Keep 20% for evaluation. Stratification keeps the class balance; a fixed seed makes the split repeatable. Both models use exactly the same rows for a fair comparison.

In [ ]:
# Use the same split for both models. The seed makes it repeatable.
random_state = 42
test_size = 0.2
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=random_state, stratify=y
)

# Record these shared settings in both MLflow runs.
split_params = {
    "dataset": "iris",
    "test_size": test_size,
    "split_random_state": random_state,
    "stratify": True,
}
print("Training rows:", len(X_train), "Evaluation rows:", len(X_test))

## 5. Train and track Logistic Regression
A run groups the settings, evaluation result, and saved model. Accuracy is the fraction of correct predictions. The model signature describes its inputs and outputs; a small input example helps inspect the artifact.

In [ ]:
logistic_model = LogisticRegression(max_iter=200, random_state=random_state)

# Everything inside this block belongs to one training run.
with mlflow.start_run(run_name="logistic-regression") as logistic_run:
    # Train on the training rows and evaluate on held-out rows.
    logistic_model.fit(X_train, y_train)
    logistic_predictions = logistic_model.predict(X_test)
    logistic_accuracy = accuracy_score(y_test, logistic_predictions)

    # Parameters describe the settings; the metric records the result.
    mlflow.log_params(split_params)
    mlflow.log_params({
        "model_type": "LogisticRegression",
        "max_iter": logistic_model.max_iter,
        "C": logistic_model.C,
        "solver": logistic_model.solver,
        "random_state": logistic_model.random_state,
    })
    mlflow.log_metric("accuracy", logistic_accuracy)

    # Save the model, its input/output schema, and a small input example.
    logistic_model_info = mlflow.sklearn.log_model(
        sk_model=logistic_model,
        name="model",
        signature=infer_signature(X_test, logistic_predictions),
        input_example=X_train.head(3),
    )

# Keep the run ID so we can compare this attempt later.
logistic_run_id = logistic_run.info.run_id
print("Run:", logistic_run_id, "Accuracy:", logistic_accuracy)

## 6. Train and track Random Forest
The second approach gets a separate run in the same experiment. Log the same accuracy metric and split settings so results remain comparable.

In [ ]:
forest_model = RandomForestClassifier(n_estimators=100, random_state=random_state)

# Start a separate run in the same experiment.
with mlflow.start_run(run_name="random-forest") as forest_run:
    # Use the same training and evaluation rows as Logistic Regression.
    forest_model.fit(X_train, y_train)
    forest_predictions = forest_model.predict(X_test)
    forest_accuracy = accuracy_score(y_test, forest_predictions)

    mlflow.log_params(split_params)
    mlflow.log_params({
        "model_type": "RandomForestClassifier",
        "n_estimators": forest_model.n_estimators,
        "max_depth": forest_model.max_depth,
        "random_state": forest_model.random_state,
    })
    mlflow.log_metric("accuracy", forest_accuracy)

    # Save the forest we just trained, including its tree objects.
    forest_model_info = mlflow.sklearn.log_model(
        sk_model=forest_model,
        name="model",
        skops_trusted_types=["sklearn.tree._tree.Tree"],
        signature=infer_signature(X_test, forest_predictions),
        input_example=X_train.head(3),
    )

forest_run_id = forest_run.info.run_id
print("Run:", forest_run_id, "Accuracy:", forest_accuracy)

## 7. Compare the logged runs
Read results back from MLflow, rather than relying on in-memory accuracy variables. Restrict comparison to this execution's two run IDs so older workshop runs cannot accidentally win.

In [ ]:
# Read back only the two runs created in this notebook execution.
logistic_result = client.get_run(logistic_run_id)
forest_result = client.get_run(forest_run_id)

# Put the logged results and saved model locations in one table.
comparison = pd.DataFrame([
    {
        "model_type": logistic_result.data.params["model_type"],
        "accuracy": logistic_result.data.metrics["accuracy"],
        "run_id": logistic_run_id,
        "model_uri": logistic_model_info.model_uri,
    },
    {
        "model_type": forest_result.data.params["model_type"],
        "accuracy": forest_result.data.metrics["accuracy"],
        "run_id": forest_run_id,
        "model_uri": forest_model_info.model_uri,
    },
])
comparison[["model_type", "accuracy", "run_id"]]

## 8. Select the winner
The highest accuracy wins. If both models tie, choose Logistic Regression, the first row in our table. This small holdout demonstrates selection; a real evaluation should reserve an independent final test set.

In [ ]:
# idxmax finds the row with the highest logged accuracy.
# For a tie, it picks the first row: Logistic Regression.
winner = comparison.loc[comparison["accuracy"].idxmax()]
winner_model_uri = winner["model_uri"]

print("Winner:", winner["model_type"])
print("Accuracy:", winner["accuracy"])
print("Source run:", winner["run_id"])

## 9. Register the winning model
Registration creates a version under the shared name `iris-classifier`, linked to its logged model and source run. We register the already saved model; we do not retrain it. Each full rerun adds a new version.

In [ ]:
# Register the saved winner without training it again.
registered_model_name = "iris-classifier"
model_version = mlflow.register_model(
    model_uri=winner_model_uri,
    name=registered_model_name,
)
print("Registered:", model_version.name, "Version:", model_version.version)

## 10. Assign and verify the candidate alias
An alias is a movable name for a model version. `candidate` now points to the winner from this execution. It records our selection without deploying anything. Earlier versions stay in the registry when this alias moves.

In [ ]:
# Point the candidate alias at the newly registered version.
client.set_registered_model_alias(
    name=registered_model_name,
    alias="candidate",
    version=model_version.version,
)

# Read the alias back to confirm where it points.
candidate = client.get_model_version_by_alias(registered_model_name, "candidate")
print("Candidate version:", candidate.version)
print("Source run:", candidate.run_id)

## 11. Load the candidate to verify the saved model
Resolve the alias and check that the stored model reproduces the selected accuracy. This is a local verification of the model artifact, not a deployment.

In [ ]:
# Load the saved model by alias instead of a fixed version number.
candidate_model = mlflow.sklearn.load_model(f"models:/{registered_model_name}@candidate")
candidate_accuracy = accuracy_score(y_test, candidate_model.predict(X_test))

# Confirm the saved model belongs to the winner and reproduces its score.
assert candidate.run_id == winner["run_id"]
assert candidate_accuracy == winner["accuracy"]
print("Reloaded candidate accuracy:", candidate_accuracy)

## Next: let Airflow orchestrate the same workflow
We have loaded data, trained and tracked two models, compared their accuracy, registered the winner, and assigned `candidate`.

Airflow will later coordinate these steps as dependent tasks, with scheduling, retries, and execution visibility. MLflow will continue recording the runs and managing model versions. No deployment or additional infrastructure is needed for today's notebook.

To inspect the results, activate the environment in a terminal at the project root and run:

```sh
python -m mlflow ui --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000
```

Open http://127.0.0.1:5000. See the README for setup and rerun behavior.